<a href="https://colab.research.google.com/github/TheProgrammingMinistry/F1-GrandPrix-Prediction-model/blob/main/Azerbaijan_Grand_Prix_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install fastf1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.0/136.0 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.8/70.8 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 44.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.6/55.6 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 424.7/424.7 kB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.8/74.8 kB 4.9 MB/s eta 0:00:00
  Attempting uninstall: msgpack
    Found existing installation: msgpack 1.2.2
    Uninstalling msgpack-1.2.2:
      Successfully uninstalled msgpack-1.2.2


In [ ]:
!pip install fastf1
import argparse
import os
import warnings

import fastf1
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")

# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------

CACHE_DIR = "./f1_cache"
DATA_PATH = "./race_dataset.csv"
MODEL_FEATURES = [
    "grid_position",
    "driver_form_last5",
    "team_form_last5",
    "driver_circuit_avg_finish",
    "driver_dnf_rate",
    "season_points_so_far",
    "constructor_points_so_far",
]
TARGET = "finish_position"

# Seasons used to build the training set. FastF1 has clean data back to ~2018.
SEASONS = [2022, 2023] # Corrected to only include available historical seasons

# The race you want to predict. Update this to whatever is next on the
# calendar — as of writing, the next round after Madrid (Sep 11-13) is
# the Azerbaijan GP in Baku (Sep 24-26, 2026).
NEXT_RACE = {"year": 2023, "event_name": "Azerbaijan Grand Prix"} # Changed to a historical race for demonstration purposes

os.makedirs(CACHE_DIR, exist_ok=True)
fastf1.Cache.enable_cache(CACHE_DIR)


# ---------------------------------------------------------------------------
# 1. Data collection
# ---------------------------------------------------------------------------

def fetch_season_results(year: int) -> pd.DataFrame:
    """Pull race + qualifying results for every completed round in a season."""
    schedule = fastf1.get_event_schedule(year, include_testing=False)
    rows = []

    for _, event in schedule.iterrows():
        round_num = event["RoundNumber"]
        event_name = event["EventName"]

        try:
            race = fastf1.get_session(year, round_num, "R")
            race.load(laps=False, telemetry=False, weather=False, messages=False)
        except Exception as e:
            # Race hasn't happened yet, or session data isn't published.
            print(f"skip {year} R{round_num} ({event_name}): {e}")
            continue

        try:
            quali = fastf1.get_session(year, round_num, "Q")
            quali.load(laps=False, telemetry=False, weather=False, messages=False)
            grid_lookup = quali.results.set_index("Abbreviation")["Position"]
        except Exception:
            grid_lookup = race.results.set_index("Abbreviation")["GridPosition"]

        results = race.results.copy()
        for _, r in results.iterrows():
            rows.append({
                "season": year,
                "round": round_num,
                "event_name": event_name,
                "driver": r["Abbreviation"],
                "team": r["TeamName"],
                "grid_position": grid_lookup.get(r["Abbreviation"], r["GridPosition"]),
                "finish_position": r["Position"],
                "status": r["Status"],       # "Finished", "Accident", "+1 Lap", etc.
                "points": r["Points"],
            })

    return pd.DataFrame(rows)


def build_dataset():
    all_seasons = []
    for year in SEASONS:
        print(f"Fetching {year} season...")
        all_seasons.append(fetch_season_results(year))
    df = pd.concat(all_seasons, ignore_index=True)
    df.to_csv(DATA_PATH, index=False)
    print(f"Saved {len(df)} driver-race rows to {DATA_PATH}")
    return df


# ---------------------------------------------------------------------------
# 2. Feature engineering
# ---------------------------------------------------------------------------

def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Turn raw per-race results into predictive features. Everything is
    computed using only PAST races relative to each row, so there's no
    leakage of future information into training.
    """
    df = df.sort_values(["season", "round"]).reset_index(drop=True)

    # DNF flag (anything other than "Finished" or a lapped-but-classified finish)
    df["dnf"] = ~df["status"].str.contains(r"Finished|\+\d+ Lap", regex=True, na=False)

    df["driver_form_last5"] = np.nan
    df["team_form_last5"] = np.nan
    df["driver_circuit_avg_finish"] = np.nan
    df["driver_dnf_rate"] = np.nan
    df["season_points_so_far"] = np.nan
    df["constructor_points_so_far"] = np.nan

    for driver, grp in df.groupby("driver"):
        idx = grp.index
        df.loc[idx, "driver_form_last5"] = (
            grp["finish_position"].shift(1).rolling(5, min_periods=1).mean().values
        )
        df.loc[idx, "driver_dnf_rate"] = (
            grp["dnf"].shift(1).rolling(10, min_periods=1).mean().values
        )

    for team, grp in df.groupby("team"):
        idx = grp.index
        df.loc[idx, "team_form_last5"] = (
            grp["finish_position"].shift(1).rolling(10, min_periods=1).mean().values
        )
        df.loc[idx, "constructor_points_so_far"] = (
            grp.groupby("season")["points"].cumsum().shift(1).values
        )

    for (driver, event), grp in df.groupby(["driver", "event_name"]):
        idx = grp.index
        df.loc[idx, "driver_circuit_avg_finish"] = (
            grp["finish_position"].shift(1).expanding().mean().values
        )

    for (driver, season), grp in df.groupby(["driver", "season"]):
        idx = grp.index
        df.loc[idx, "season_points_so_far"] = grp["points"].cumsum().shift(1).values

    # Fill early-career / early-season gaps with sane defaults
    fill_values = {
        "driver_form_last5": df["finish_position"].mean(),
        "team_form_last5": df["finish_position"].mean(),
        "driver_circuit_avg_finish": df["finish_position"].mean(),
        "driver_dnf_rate": df["dnf"].mean(),
        "season_points_so_far": 0,
        "constructor_points_so_far": 0,
    }
    df = df.fillna(fill_values)

    return df


# ---------------------------------------------------------------------------
# 3. Train / evaluate
# ---------------------------------------------------------------------------

def train_model(df: pd.DataFrame):
    df = engineer_features(df)
    df = df.dropna(subset=["grid_position", TARGET])

    X = df[MODEL_FEATURES]
    y = df[TARGET].astype(float)

    # Chronological split: train on earlier rounds, test on the most recent
    # ones. A random split would leak future form into the past.
    split_point = int(len(df) * 0.85)
    df_sorted = df.sort_values(["season", "round"])
    train_idx = df_sorted.index[:split_point]
    test_idx = df_sorted.index[split_point:]

    X_train, X_test = X.loc[train_idx], X.loc[test_idx]
    y_train, y_test = y.loc[train_idx], y.loc[test_idx]

    model = GradientBoostingRegressor(
        n_estimators=300,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.8,
        random_state=42,
    )
    model.fit(X_train, y_train)

    preds = model.predict(X_test)
    mae = mean_absolute_error(y_test, preds)
    print(f"Test MAE (avg positions off): {mae:.2f}")

    importances = pd.Series(model.feature_importances_, index=MODEL_FEATURES)
    print("\nFeature importance:")
    print(importances.sort_values(ascending=False).to_string())

    return model, df


# ---------------------------------------------------------------------------
# 4. Predict the next race
# ---------------------------------------------------------------------------

def predict_next_race(model, df: pd.DataFrame, year: int, event_name: str):
    """
    Build a feature row per current-grid driver for the target race using
    their most recent known form, then rank by predicted finish position.

    Grid position isn't known until qualifying, so this uses each driver's
    average grid position over their last 3 races as a stand-in. Re-run
    this step after qualifying (plug in the real grid) for a sharper call.
    """
    latest_season = df[df["season"] == year]
    if latest_season.empty:
        latest_season = df[df["season"] == df["season"].max()]

    drivers = latest_season["driver"].unique()
    rows = []
    for driver in drivers:
        driver_rows = df[df["driver"] == driver].sort_values(["season", "round"])
        if driver_rows.empty:
            continue
        last = driver_rows.iloc[-1]
        recent_grid = driver_rows["grid_position"].tail(3).mean()

        rows.append({
            "driver": driver,
            "team": last["team"],
            "grid_position": recent_grid,
            "driver_form_last5": last["driver_form_last5"],
            "team_form_last5": last["team_form_last5"],
            "driver_circuit_avg_finish": df[
                (df["driver"] == driver) & (df["event_name"] == event_name)
            ]["finish_position"].mean() if not df[
                (df["driver"] == driver) & (df["event_name"] == event_name)
            ].empty else last["driver_circuit_avg_finish"],
            "driver_dnf_rate": last["driver_dnf_rate"],
            "season_points_so_far": last["season_points_so_far"] + last["points"],
            "constructor_points_so_far": last["constructor_points_so_far"] + last["points"],
        })

    pred_df = pd.DataFrame(rows).fillna(df[MODEL_FEATURES].mean())
    pred_df["predicted_position"] = model.predict(pred_df[MODEL_FEATURES])
    pred_df = pred_df.sort_values("predicted_position").reset_index(drop=True)
    pred_df.index += 1

    print(f"\nPredicted order — {event_name} {year}")
    print(pred_df[["driver", "team", "predicted_position"]].to_string())

    return pred_df


# ---------------------------------------------------------------------------
# CLI
# ---------------------------------------------------------------------------

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--build-dataset", action="store_true")
    parser.add_argument("--train", action="store_true")
    parser.add_argument("--predict", action="store_true")
    parser.add_argument("--all", action="store_true")
    args, unknown = parser.parse_known_args()

    if args.all or args.build_dataset:
        df = build_dataset()
    else:
        df = pd.read_csv(DATA_PATH) if os.path.exists(DATA_PATH) else None

    if args.all or args.train:
        model, df = train_model(df)

    if args.all or args.predict:
        predict_next_race(model, df, NEXT_RACE["year"], NEXT_RACE["event_name"])

In [ ]:
# 1. Make sure you've already run the cells that create `df` and `model`
#    (sections 3-5 in the notebook). Those give you a trained model and
#    a dataframe of historical results with features attached.

# Fix: Explicitly call build_dataset and train_model to define df and model
df = build_dataset()
model, df = train_model(df)

# 2. Pick the race you want to predict
year = 2026
event_name = "Azerbaijan Grand Prix"

# 3. Call the prediction function defined in section 6
predicted_order = predict_next_race(model, df, year, event_name)

# 4. Look at the result
predicted_order[["driver", "team", "predicted_position"]]

Fetching 2022 season...


core           INFO 	Loading data for Bahrain Grand Prix - Race [v3.8.3]
INFO:fastf1.fastf1.core:Loading data for Bahrain Grand Prix - Race [v3.8.3]
req            INFO 	No cached data found for session_info. Loading data...
INFO:fastf1.fastf1.req:No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
INFO:fastf1.api:Fetching session info data...
DEBUG:fastf1.api:Falling back to livetiming mirror (https://livetiming-mirror.fastf1.dev)
req            INFO 	Data has been written to cache!
INFO:fastf1.fastf1.req:Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
INFO:fastf1.fastf1.req:No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
INFO:fastf1.api:Fetching driver list...
DEBUG:fastf1.api:Falling back to livetiming mirror (https://livetiming-mirror.fastf1.dev)
req            INFO 	Data has been written to cache!
INFO:fastf1.fastf1.req

Fetching 2023 season...


INFO:fastf1.fastf1.req:No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
INFO:fastf1.api:Fetching session info data...
DEBUG:fastf1.api:Falling back to livetiming mirror (https://livetiming-mirror.fastf1.dev)
logger      WARNING 	Failed to load session info data!
DEBUG:fastf1.fastf1.core:Traceback for failure in session info data
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/fastf1/logger.py", line 112, in __wrapped
    return func(*args, **kwargs)
  File "/usr/local/lib/python3.13/dist-packages/fastf1/core.py", line 1452, in _load_session_info
    self._session_info = api.session_info(self.api_path,
                         ~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^
                                          livedata=livedata)
                                          ^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/fastf1/req.py", line 459, in _cached_api_request
    data = func(api_pat

Saved 880 driver-race rows to ./race_dataset.csv
Test MAE (avg positions off): 3.56

Feature importance:
team_form_last5              0.329976
grid_position                0.313963
driver_form_last5            0.118198
constructor_points_so_far    0.082276
season_points_so_far         0.078095
driver_circuit_avg_finish    0.049865
driver_dnf_rate              0.027628

Predicted order — Azerbaijan Grand Prix 2026
   driver            team  predicted_position
1     VER        Red Bull            4.866515
2     LEC         Ferrari            5.511514
3     ALO    Aston Martin            6.899289
4     HAM        Mercedes            7.405447
5     NOR         McLaren            8.228615
6     GAS  Alpine F1 Team            8.639301
7     PER        Red Bull            9.692352
8     PIA         McLaren           10.013295
9     OCO  Alpine F1 Team           10.645950
10    SAI         Ferrari           11.671722
11    RUS        Mercedes           11.816580
12    ALB        Williams      

,driver,team,predicted_position
1,VER,Red Bull,4.866515
2,LEC,Ferrari,5.511514
3,ALO,Aston Martin,6.899289
4,HAM,Mercedes,7.405447
5,NOR,McLaren,8.228615
6,GAS,Alpine F1 Team,8.639301
7,PER,Red Bull,9.692352
8,PIA,McLaren,10.013295
9,OCO,Alpine F1 Team,10.645950
10,SAI,Ferrari,11.671722
